# 07 — CW Diagnostics

Legacy experiment 3.

Continuous wave output on all quantum elements for spectrum analyzer verification.
Use this notebook to verify RF chain integrity before proceeding with pulsed experiments.

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    continuous_wave,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="07_cw_diagnostics",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

prev_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if prev_checkpoint is not None:
    print(f"Prior stage 06 status: {prev_checkpoint['status']}")

## 2. Configure CW Target Elements

Select which quantum elements to drive with CW tones for SA verification.

In [ ]:
CW_ELEMENTS = ["qubit", "readout", "storage"]

print("CW target elements:")
for elem in CW_ELEMENTS:
    print(f"  - {elem}")

## 3. Run CW Program — Exp 3

Output continuous-wave tones on all configured elements.
Use a spectrum analyzer to verify the RF chain.

> **Note**: This cell blocks while the CW program is running.
> Stop the cell (interrupt kernel) when SA verification is complete.

In [ ]:
continuous_wave(session, elements=CW_ELEMENTS)

print("CW program stopped.")

## 4. Manual SA Verification

**Checklist:**
- [ ] Qubit drive tone visible at expected frequency
- [ ] Readout tone visible at expected frequency
- [ ] Storage tone visible at expected frequency
- [ ] No unexpected spurs or harmonics
- [ ] Power levels match expected values

## 5. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="07_cw_diagnostics",
    status="characterized",
    summary="CW diagnostics: continuous wave output verified on all elements via spectrum analyzer.",
    consumed_inputs={
        "cw_elements": CW_ELEMENTS,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="08_pulse_waveform_definition",
    notes=[
        "Manual SA verification — no automated analysis.",
        "continuous_wave is a legacy proxy utility.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")